<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/validacion_cruzada_g.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evitando Overfitting con Cross Validation
## Usando K-Fold para mejorar el split train/test

### 1. El temido problema de Overfitting

Uno de los errores más frecuentes (y costosos) al desarrollar modelos de machine learning es confiar ciegamente en un buen resultado en un único conjunto de entrenamiento.

Imagina que entrenas un modelo y obtienes:

- Accuracy de entrenamiento: **98.7%**  
- Accuracy en el conjunto de test: **72.4%**

Esto es el clásico signo de **overfitting**: el modelo ha aprendido demasiado bien los detalles, particularidades e incluso ruido del conjunto de entrenamiento, pero no ha capturado patrones generales que funcionen en datos nuevos.

¿Por qué ocurre con tanta frecuencia un split train/test simple?

- El conjunto de test puede ser, por casualidad, más fácil o más difícil que el promedio real  
- En datasets pequeños, una mala partición puede hacer que perdamos información valiosa o que evaluemos en un subconjunto no representativo  
- Si hacemos muchas pruebas y ajustes usando siempre el mismo conjunto de test, terminamos haciendo **data leakage indirecto** (ajustamos hiperparámetros “mirando” el test varias veces)

La solución ideal sería tener tres conjuntos bien separados:

- **train** → para ajustar pesos  
- **validation** → para elegir modelo / hiperparámetros  
- **test** → para estimación final e imparcial

Pero cuando los datos son limitados, reservar un conjunto de validación fijo reduce aún más el tamaño del entrenamiento. Aquí entra la validación cruzada.

### 2. ¿Qué es la validación cruzada?

La validación cruzada (cross-validation, CV) es una técnica que nos permite **usar todos los datos disponibles tanto para entrenar como para evaluar**, pero de forma ordenada y sin hacer trampa.

La idea central es muy simple:

Realizamos **varios experimentos** diferentes, cada uno con una partición distinta de los datos. En cada experimento:

- Entrenamos con la mayor parte de los datos  
- Evaluamos con la parte que dejamos fuera  
- Repetimos el proceso varias veces con diferentes divisiones

Luego promediamos los resultados (y calculamos la dispersión) para obtener una estimación **mucho más robusta y menos dependiente de la suerte** de cómo se dividieron los datos.

La versión más popular y la que casi todo el mundo usa como punto de partida es **K-Fold Cross Validation**:

1. Se divide el conjunto completo en **K subconjuntos** (folds) de tamaño similar  
2. Se realizan **K iteraciones** (o “folds”)  
3. En cada iteración:  
   - se usa 1 fold como conjunto de **validación**  
   - se usan los K−1 folds restantes para **entrenar**  
4. Se guarda la métrica de evaluación obtenida en ese fold de validación  
5. Al final se calcula:  
   - **promedio** de las K métricas → estimación central del rendimiento esperado  
   - **desviación estándar** → nos indica la estabilidad del modelo frente a diferentes particiones

Ventajas clave respecto a un único train/test split:

- Todos los ejemplos se usan para entrenar y para validar (ninguno se desperdicia)  
- La estimación del error es mucho menos sesgada y menos variable  
- Podemos detectar modelos que son muy sensibles a la partición de datos

### 3. Cómo funciona realmente K-Fold paso a paso (sin código)

Imagina que tenemos 100 ejemplos y elegimos K = 5 (una de las opciones más comunes).

El proceso se ve así:

```
Fold 1:  train [1–80]   →  valid [81–100]   →  calculamos métrica A
Fold 2:  train [1–40, 81–100] → valid [41–80]   →  métrica B
Fold 3:  train [1–20, 41–100] → valid [21–40]   →  métrica C
Fold 4:  train [21–100]       → valid [1–20]    →  métrica D
Fold 5:  train [41–100, 1–20] → valid [21–40]   →  métrica E   (el orden real varía)
```

(En la práctica se suele barajar antes los datos para que los folds no sigan el orden original)

Después de las 5 iteraciones tenemos 5 métricas diferentes. Entonces:

**Rendimiento esperado ≈ promedio de las 5 métricas**  
**Incertidumbre ≈ desviación estándar de las 5 métricas**

Interpretaciones útiles:

- Si la desviación estándar es muy alta → el modelo es muy sensible a qué datos ve durante el entrenamiento (señal de inestabilidad o posible overfitting severo en algunos subconjuntos)  
- Si el promedio es mucho peor que el resultado que obtenías con un simple train/test → probablemente estabas sobreestimando el rendimiento real  
- Si el promedio es similar al de train pero con alta varianza → el modelo podría estar memorizando en algunos folds y generalizando mal en otros

Elegir el valor de K  
- K = 5 → buen balance entre tiempo de cómputo y estabilidad  
- K = 10 → estimación más precisa, pero ~2× más lento  
- K = “leave-one-out” (K = n) → muy preciso pero extremadamente lento (solo se usa en datasets muy pequeños)

### 4. Variantes importantes: StratifiedKFold y GroupKFold

La división aleatoria simple (KFold) funciona bien en regresión y en clasificación con clases muy balanceadas. Pero falla en varios casos reales comunes.

**StratifiedKFold**  
Mantiene (aproximadamente) la **misma proporción de cada clase** en cada fold que en el conjunto completo.

Ejemplo: si tienes 80% clase 0 y 20% clase 1, cada fold de validación debería tener ~20% de clase 1.

**Cuándo es casi obligatorio usarlo**:

- Clasificación binaria o multiclase con desbalance moderado o severo  
- Datasets con pocas muestras por clase minoritaria  
- Cualquier problema donde la distribución de clases importe mucho

**GroupKFold**  
No mezcla muestras que pertenecen al **mismo grupo** entre entrenamiento y validación.

Casos típicos:

- Múltiples registros del mismo paciente / usuario / dispositivo  
- Varias fotos del mismo individuo  
- Mediciones repetidas del mismo sensor / día / localización  
- Series temporales donde no queremos leakage entre periodos cercanos

En estos casos un KFold o StratifiedKFold normal puede generar **data leakage grave** porque el modelo ve datos “futuros” o “del mismo sujeto” durante el entrenamiento.

Otras variantes útiles (mención breve):

- RepeatedKFold / RepeatedStratifiedKFold → repite el proceso varias veces con diferentes barajados  
- TimeSeriesSplit → para datos ordenados temporalmente (importante en forecasting)

Elegir la estrategia de validación cruzada correcta es una de las decisiones que más impacto tiene en la fiabilidad de tus resultados finales.

---

### 5. Ejemplo práctico con scikit-learn y el dataset Heart Disease (≈1,190 muestras)

Vamos a aplicar la validación cruzada a un problema real de clasificación médica: predecir la presencia de enfermedad cardíaca a partir de características clínicas.

Este dataset combina varias fuentes del UCI Heart Disease (Cleveland + Hungary + otros), tiene 1,190 observaciones y 13 características + target binario.

Primero cargamos los datos (asumimos que estás en Kaggle o has subido el archivo):

```python
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load the combined Heart Disease dataset
# Path example in Kaggle: /kaggle/input/heart-disease-dataset/heart.csv
df = pd.read_csv('/kaggle/input/heart-disease-dataset/heart.csv')  # adjust path if needed

X = df.drop('target', axis=1)
y = df['target']

print("Dataset shape:", X.shape)
print("Features:", X.columns.tolist())
print("\nClass distribution:")
print(y.value_counts(normalize=True).round(4))
```

### Comparación: Split simple vs K-Fold

Primero veamos el comportamiento típico con un único split train/test:

```python
# Single train-test split (for comparison)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc  = accuracy_score(y_test,  model.predict(X_test))

print(f"Single split → Train accuracy: {train_acc:.4f}")
print(f"Single split → Test  accuracy:  {test_acc:.4f}")
```

Ahora aplicamos K-Fold Cross-Validation (sin estratificar primero):

```python
# 5-Fold CV (non-stratified)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores_kf = cross_val_score(
    DecisionTreeClassifier(max_depth=5, random_state=42),
    X, y,
    cv=kf,
    scoring='accuracy'
)

print("KFold (5) scores:     ", [f"{s:.4f}" for s in scores_kf])
print(f"Mean: {scores_kf.mean():.4f}  ± {scores_kf.std():.4f}")
```

### 6. Interpretación de resultados

Con este dataset más grande, los resultados suelen ser más estables que con Iris o Cleveland solo.

Ejemplo típico de salida que podrías observar:

```
KFold (5) scores:      ['0.8151', '0.8319', '0.8151', '0.8151', '0.8487']
Mean: 0.8252  ± 0.0130
```

Qué nos dice esto:

- Media ≈ 0.82–0.83 → rendimiento realista esperado (un árbol simple no es muy potente aquí)  
- Desviación estándar baja (~0.01–0.02) → el modelo es bastante estable frente a diferentes particiones  
- El accuracy en **train** del split simple suele ser mucho más alto (0.90–0.97) → claro signo de overfitting  
- La diferencia entre train y CV/test muestra por qué confiar solo en train o en un único test es peligroso

Ahora veamos la versión recomendada para clasificación: **StratifiedKFold**

```python
# Stratified 5-Fold CV (recommended)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_strat = cross_val_score(
    DecisionTreeClassifier(max_depth=5, random_state=42),
    X, y,
    cv=skf,
    scoring='accuracy'
)

print("StratifiedKFold scores:", [f"{s:.4f}" for s in scores_strat])
print(f"Mean: {scores_strat.mean():.4f}  ± {scores_strat.std():.4f}")
```

En este dataset el desbalance es leve (~53% / 47%), por lo que la diferencia entre KFold y StratifiedKFold suele ser pequeña. Sin embargo, en problemas reales de medicina (donde el desbalance puede ser 80–90% vs 10–20%) la estratificación marca una diferencia muy grande en la fiabilidad de las métricas.

### 7. Errores comunes a evitar

Algunos errores frecuentes que aparecen en notebooks y competiciones:

1. Usar KFold simple en clasificación con clases desbalanceadas → folds con 0 ejemplos de la clase minoritaria  
2. Realizar feature engineering o imputación sobre todo el dataset antes de CV → leakage severo  
3. Ajustar hiperparámetros mirando repetidamente el mismo conjunto de test fijo → optimización sobre test  
4. Reportar solo la media sin desviación estándar → oculta inestabilidad  
5. Usar cross_val_score para comparar modelos, pero entrenar el modelo final sin CV → inconsistencia  
6. Olvidar shuffle=True cuando los datos tienen orden (ej. pacientes ordenados por fecha de ingreso)  
7. Usar K muy alto (ej. 20) en datasets medianos → tiempo excesivo sin ganancia significativa  
8. No considerar métricas más adecuadas que accuracy en problemas médicos (recall, F1, AUC-PR)

### 8. Conclusiones

- Un único **train/test split** es rápido, pero ruidoso y puede sobreestimar o subestimar gravemente el rendimiento real.  
- **K-Fold Cross Validation**, y especialmente **StratifiedKFold** en clasificación, ofrece una estimación mucho más robusta y honesta.  
- La combinación de **media + desviación estándar** nos da información sobre el nivel esperado y la estabilidad del modelo.  
- En problemas médicos/farmacéuticos (como predicción de riesgo cardiovascular), elegir la estrategia de validación adecuada es crítico para evitar conclusiones erróneas que podrían afectar decisiones clínicas o de desarrollo de fármacos.

**Recomendación práctica para la mayoría de proyectos:**

- Validación principal → **StratifiedKFold(n_splits=5)** o **10**  
- Selección de modelo y tuning → nested cross-validation o validación interna  
- Conjunto hold-out final → reservar ~20% solo al final, para evaluación imparcial del modelo ganador

¡Espero que este notebook te anime a aplicar validación cruzada en tus proyectos!